## 1. Import the required libraries

- `pandas` to store the support chats in a DataFrame.
- `string` to get Python's built-in punctuation characters.
- `re` to clean extra spaces.
- `nltk` for stop words, Porter stemming, and edit distance.
- `spacy` for lemmatization.



In [25]:
import pandas as pd
import string
import re
import nltk
import spacy

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer



nlp = spacy.load("en_core_web_sm")

nltk_stop_words = set(stopwords.words("english"))
spacy_stop_words = nlp.Defaults.stop_words

print("Libraries loaded successfully.")

Libraries loaded successfully.


## 2. Create the support chat data



In [26]:
support_chats = [

    "Sir my pakage has not arived yet, can you chek the delivery status?",

    "I beleive my marks are not updated in the portal yet, can you chek?",

    "The LMS is crashin when I try to submit my lab assignment.",

    "I definately need help with my data science assingment before the deadline.",

    "Can you tell me wich room our NLP class is in?",

    "I recieved the wrong marks for my lab submission, please chek them.",

    "The exam schedule has not been updted yet, and I don't know the delievery date.",

    "Please chek why my attendance is not showing on the LMS.",

    "I definately need help with my DSA assignment submission.",

    "The submission deadline is seperate for each section, but the LMS shows the wrong date.",

    "I recieved an error while uploading my Jupyter notebook and the LMS crashin again.",

    "Can you tell me wich dataset we have to use for the lab?",

    "My quizz marks have not been added yet, I beleive there is an error.",

    "I beleive the timetable has been changed but the LMS is not updated.",

    "The portal is crashin when I try to download my result and an error occured."
]

print("Number of support chats:", len(support_chats))

for i, chat in enumerate(support_chats, 1):
    print(f"{i}. {chat}")

Number of support chats: 15
1. Sir my pakage has not arived yet, can you chek the delivery status?
2. I beleive my marks are not updated in the portal yet, can you chek?
3. The LMS is crashin when I try to submit my lab assignment.
4. I definately need help with my data science assingment before the deadline.
5. Can you tell me wich room our NLP class is in?
6. I recieved the wrong marks for my lab submission, please chek them.
7. The exam schedule has not been updted yet, and I don't know the delievery date.
8. Please chek why my attendance is not showing on the LMS.
9. I definately need help with my DSA assignment submission.
10. The submission deadline is seperate for each section, but the LMS shows the wrong date.
11. I recieved an error while uploading my Jupyter notebook and the LMS crashin again.
12. Can you tell me wich dataset we have to use for the lab?
13. My quizz marks have not been added yet, I beleive there is an error.
14. I beleive the timetable has been changed but th

# Home Activity 1
## Clean the support chats end to end


The percentage reduction:

**Reduction (%) = ((Vocabulary Before − Vocabulary After) / Vocabulary Before) × 100**

In [27]:


df = pd.DataFrame({"chat": support_chats})

df

def clean_text(text):
    # Lowercase and remove punctuation first
    text = text.lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\s+", " ", text).strip()

    doc = nlp(text)

    cleaned_words = []
    for token in doc:
        if token.text not in spacy_stop_words and token.text.strip() != "":
            cleaned_words.append(token.lemma_)

    return " ".join(cleaned_words)

# Vocabulary before cleaning
words_before = " ".join(df["chat"]).lower().split()
vocab_before = len(set(words_before))

# Apply the complete cleaning function
df["cleaned_chat"] = df["chat"].apply(clean_text)

# Vocabulary after cleaning
words_after = " ".join(df["cleaned_chat"]).split()
vocab_after = len(set(words_after))

reduction = ((vocab_before - vocab_after) / vocab_before) * 100

print("Vocabulary before cleaning:", vocab_before)
print("Vocabulary after cleaning:", vocab_after)
print(f"Vocabulary reduction: {reduction:.2f}%")

df[["chat", "cleaned_chat"]]

Vocabulary before cleaning: 99
Vocabulary after cleaning: 57
Vocabulary reduction: 42.42%


,chat,cleaned_chat
0,"Sir my pakage has not arived yet, can you chek...",sir pakage arive chek delivery status
1,I beleive my marks are not updated in the port...,beleive mark update portal chek
2,The LMS is crashin when I try to submit my lab...,lm crashin try submit lab assignment
3,I definately need help with my data science as...,definately need help data science assingment d...
4,Can you tell me wich room our NLP class is in?,tell wich room nlp class
5,I recieved the wrong marks for my lab submissi...,recieve wrong mark lab submission chek
6,"The exam schedule has not been updted yet, and...",exam schedule updte not know delievery date
7,Please chek why my attendance is not showing o...,chek attendance show lm
8,I definately need help with my DSA assignment ...,definately need help dsa assignment submission
9,The submission deadline is seperate for each s...,submission deadline seperate section lm show w...


# Home Activity 2
## Compare Porter stemming and spaCy lemmatization

Stemming and lemmatization both try to reduce words to a simpler form, but they work differently.

**Porter stemming:** removes word endings using fixed rules. It can produce a string that is not a real English word.

**spaCy lemmatization:** tries to return the dictionary form of a word.



In [28]:
stemmer = PorterStemmer()
english_words = set(word.lower() for word in nltk.corpus.words.words())

cleaned_vocab = sorted(set(" ".join(df["cleaned_chat"]).split()))

# Pick up to 10 words from the cleaned vocabulary
selected_words = cleaned_vocab[:10]

comparison = []

for word in selected_words:
    stem = stemmer.stem(word)
    lemma = nlp(word)[0].lemma_

    # A simple check: is the generated stem present in the English word list?
    if stem in english_words:
        stem_status = "Looks like a word"
    else:
        stem_status = "May not be a real word"

    comparison.append({
        "Original Word": word,
        "Porter Stem": stem,
        "spaCy Lemma": lemma,
        "Stem Check": stem_status
    })

comparison_df = pd.DataFrame(comparison)
comparison_df

,Original Word,Porter Stem,spaCy Lemma,Stem Check
0,add,add,add,Looks like a word
1,arive,ariv,arive,May not be a real word
2,assignment,assign,assignment,Looks like a word
3,assingment,assing,assingment,May not be a real word
4,attendance,attend,attendance,Looks like a word
5,beleive,beleiv,beleive,May not be a real word
6,change,chang,change,Looks like a word
7,chek,chek,chek,May not be a real word
8,class,class,class,Looks like a word
9,crashin,crashin,crashin,May not be a real word


# Home Activity 3
## Spelling suggestion using Minimum Edit Distance

The Lab 4 manual describes **Levenshtein / Minimum Edit Distance** as the minimum number of:

- insertions,
- deletions,
- substitutions

needed to change one word into another.

For a misspelled word, we compare it with every word in a vocabulary and select the word with the smallest edit distance.


In [29]:
import nltk

lab_vocab = [
    "receive", "definitely", "occurred", "separate", "until",
    "which", "believe", "because", "achieve", "necessary"
]

# Vocabulary needed for the support-chat spelling examples
support_vocab = lab_vocab + [
    "package", "arrived", "check", "crashing", "delivery"
]

def suggest_word(misspelled, vocabulary):
    best_word = None
    best_distance = float("inf")

    for word in vocabulary:
        distance = nltk.edit_distance(misspelled, word)

        if distance < best_distance:
            best_distance = distance
            best_word = word

    return best_word, best_distance

test_words = [
    "recieve", "definately", "occured",
    "seperate", "untill", "wich"
]

suggestions = []

for word in test_words:
    correction, distance = suggest_word(word, lab_vocab)

    suggestions.append({
        "Misspelled Word": word,
        "Suggested Correction": correction,
        "Edit Distance": distance
    })

suggestions_df = pd.DataFrame(suggestions)
suggestions_df

,Misspelled Word,Suggested Correction,Edit Distance
0,recieve,receive,2
1,definately,definitely,1
2,occured,occurred,1
3,seperate,separate,1
4,untill,until,1
5,wich,which,1


In [30]:
misspelled_words = [
    "pakage", "arived", "chek", "beleive", "definately",
    "crashin", "delievery", "wich", "recieved", "occured",
    "seperate"
]

word_corrections = {}

for word in misspelled_words:
    correction, distance = suggest_word(word, support_vocab)
    word_corrections[word] = correction

correction_rows = []

for word, correction in word_corrections.items():
    distance = nltk.edit_distance(word, correction)
    correction_rows.append({
        "Misspelled Word": word,
        "Suggested Correction": correction,
        "Edit Distance": distance
    })

correction_table = pd.DataFrame(correction_rows)
correction_table

,Misspelled Word,Suggested Correction,Edit Distance
0,pakage,package,1
1,arived,arrived,1
2,chek,check,1
3,beleive,receive,2
4,definately,definitely,1
5,crashin,crashing,1
6,delievery,delivery,1
7,wich,which,1
8,recieved,receive,3
9,occured,occurred,1


### Correct the messages

The next function goes through each word in a chat message. If the word is in our misspelling dictionary, it replaces it with the suggested correction.

For words that are not misspelled, it leaves them unchanged.

In [31]:
def correct_message(message):
    words = message.split()
    corrected_words = []

    for word in words:
        # Remove punctuation temporarily so the word can be matched
        clean_word = word.lower().strip(string.punctuation)

        if clean_word in word_corrections:
            corrected = word_corrections[clean_word]

            # Keep simple punctuation from the original word
            if word.endswith("?"):
                corrected += "?"
            elif word.endswith("."):
                corrected += "."

            corrected_words.append(corrected)
        else:
            corrected_words.append(word)

    return " ".join(corrected_words)

df["corrected_chat"] = df["chat"].apply(correct_message)

df[["chat", "corrected_chat"]]

,chat,corrected_chat
0,"Sir my pakage has not arived yet, can you chek...","Sir my package has not arrived yet, can you ch..."
1,I beleive my marks are not updated in the port...,I receive my marks are not updated in the port...
2,The LMS is crashin when I try to submit my lab...,The LMS is crashing when I try to submit my la...
3,I definately need help with my data science as...,I definitely need help with my data science as...
4,Can you tell me wich room our NLP class is in?,Can you tell me which room our NLP class is in?
5,I recieved the wrong marks for my lab submissi...,I receive the wrong marks for my lab submissio...
6,"The exam schedule has not been updted yet, and...","The exam schedule has not been updted yet, and..."
7,Please chek why my attendance is not showing o...,Please check why my attendance is not showing ...
8,I definately need help with my DSA assignment ...,I definitely need help with my DSA assignment ...
9,The submission deadline is seperate for each s...,The submission deadline is separate for each s...


# Home Activity 4
## Simulate an autocorrect feature



In [32]:
message_numbers = [1, 4, 6, 9]

autocorrect_rows = []

for number in message_numbers:
    original = df.loc[number - 1, "chat"]
    corrected = df.loc[number - 1, "corrected_chat"]

    autocorrect_rows.append({
        "Message Number": number,
        "Original": original,
        "Corrected": corrected
    })

autocorrect_df = pd.DataFrame(autocorrect_rows)
autocorrect_df

,Message Number,Original,Corrected
0,1,"Sir my pakage has not arived yet, can you chek...","Sir my package has not arrived yet, can you ch..."
1,4,I definately need help with my data science as...,I definitely need help with my data science as...
2,6,I recieved the wrong marks for my lab submissi...,I receive the wrong marks for my lab submissio...
3,9,I definately need help with my DSA assignment ...,I definitely need help with my DSA assignment ...
